<a href="https://colab.research.google.com/github/MawaSC/Quantum-Simulation-on-IQM-Resonance/blob/main/02_trotter_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

from fermi_hubbard_model import (
    L,
    N_Q,
    t_h,
    U,
    mu,
    H_qubitized,
    build_circuit,
    neel_state,
    qubit_of,
    occupations_from_counts
)

1. Compare Trotterized evolution with exact evolution

We start with the Néel state and let it evolve under the Fermi–Hubbard Hamiltonian. As the simulation progresses, we track the occupation of each lattice site for both spin types.

For comparison, we calculate the exact evolution using the Hamiltonian matrix and compare it with the results from our Trotterized circuit. This helps us see how closely the circuit follows the exact result.

To focus on the error caused by the Trotter approximation, we calculate the circuit occupations directly from its statevector instead of using a limited number of measurement shots. We can look at sampling effects separately.

In [ ]:
dt        = 0.1 #0.05, 0.1, 0.15, 0.2, and 0.5, Report the RMSE, Report the total and two_qubit depth for time = 2 , time = dt * n_max
n_max     = 30
times     = np.arange(n_max + 1) * dt

H_exact = H_qubitized(L, t_h, U, mu)
psi0    = np.zeros(2**N_Q, dtype=complex)
psi0[int(''.join(map(str, init_occ)), 2)] = 1.0

nup_q, ndn_q, nup_e, ndn_e = [], [], [], []

for n_steps in range(n_max + 1):
    # --- quantum circuit ---
    qv = build_circuit(dt, n_steps, init_occ)
    counts = qv.get_measurement()
    u, d = occupations_from_counts(counts, flipped=(n_steps % 2 == 1))
    nup_q.append(u); ndn_q.append(d)

    # --- exact reference ---
    psi = la.expm(-1j * H_exact * n_steps * dt) @ psi0
    prob = np.abs(psi)**2
    p1 = np.array([sum(prob[k] for k in range(2**N_Q) if (k >> (N_Q-1-J)) & 1)
                   for J in range(N_Q)])
    nup_e.append([p1[qubit_of(i, 'u')] for i in range(L)])
    ndn_e.append([p1[qubit_of(i, 'd')] for i in range(L)])

nup_q, ndn_q = np.array(nup_q), np.array(ndn_q)
nup_e, ndn_e = np.array(nup_e), np.array(ndn_e)

rmse = np.sqrt(np.mean((np.c_[nup_q, ndn_q] - np.c_[nup_e, ndn_e])**2, axis=1))
print(f"RMSE over all {2*L} spin-orbitals:")
print(f"   at t={times[5]:.2f}: {rmse[5]:.2e}")
print(f"   at t={times[-1]:.2f}: {rmse[-1]:.2e}   <- pure Trotter error, no shot noise")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True, sharey=True)
for i, ax in enumerate(axes.flat):
    ax.plot(times, nup_q[:, i], 'o-',  ms=4, color='crimson',   label=r'$\langle n_{\uparrow}\rangle$ Simulation')
    ax.plot(times, ndn_q[:, i], 's-',  ms=4, color='royalblue', label=r'$\langle n_{\downarrow}\rangle$ Simulation')
    ax.plot(times, nup_e[:, i], '--',  color='crimson',   alpha=.55, label='exact')
    ax.plot(times, ndn_e[:, i], '--',  color='royalblue', alpha=.55)
    ax.set_title(f"site {i}"); ax.grid(alpha=.3); ax.set_ylim(-0.05, 1.05)
axes[0,0].legend(fontsize=8, loc='center right')
for ax in axes[1]: ax.set_xlabel(r"time  $t\,/\,t_h^{-1}$")
for ax in axes[:,0]: ax.set_ylabel("occupation")
fig.suptitle(f"1D Fermi-Hubbard quench from a Neel state,  L={L},  U/t_h={U/t_h:.0f}")
fig.tight_layout()

# 2. Save as PNG (300 DPI for crisp resolution)
plt.savefig(
    'fermi_hubbard_neel_quench.png',
    bbox_inches='tight',
    dpi=300
)

# 3. Save as PDF (Vector graphics for Overleaf / LaTeX papers)
plt.savefig(
    'fermi_hubbard_neel_quench.pdf',
    bbox_inches='tight',
    dpi=300
)

plt.show()

In [ ]:
def run_simulation(dt, n_max):

  #Physical times corresponding to each Trotter step
  times = np.arange(n_max + 1) * dt

  #nup_q -> quantum/Trotterized spin-up occupations
  #ndn_q -> quantum/Trotterized spin-down occupations
  nup_q, ndn_q = [], []
  #nup_e -> exact spin-up occupations
  #ndn_e -> exact spin-down occupations
  nup_e, ndn_e = [], []

  #Loop over the Trotter steps
  for n_steps in range(n_max + 1):

    #init_occ -> Initial occupation
    qv = build_circuit(dt, n_steps, init_occ)

    counts = qv.get_measurement()

    u, d = occupations_from_counts(
        counts,
        flipped = (n_steps % 2 == 1)
    )
    #If even, False
    #If odd, True

    nup_q.append(u)
    ndn_q.append(d)

    #Time evolution operator to initial state
    psi = la.expm(-1j * H_exact * n_steps * dt) @ psi0

    #Gives probability of each computational-basis state
    prob = np.abs(psi) ** 2

    #Single qubit occupation probabilities
    p1 = np.array([
        sum(
            prob[k]
            for k in range(2**N_Q)
            if(k >> (N_Q-1-J)) & 1
        )
        for J in range(N_Q)
    ])

    nup_e.append([
        p1[qubit_of(i,'u')]
        for i in range(L)
    ])

    ndn_e.append([
        p1[qubit_of(i,'d')]
        for i in range(L)
    ])

  nup_q = np.array(nup_q)
  ndn_q = np.array(ndn_q)


  nup_e = np.array(nup_e)
  ndn_e = np.array(ndn_e)

  rmse = np.sqrt(
      np.mean(
          (np.c_[nup_q, ndn_q] - np.c_[nup_e, ndn_e])**2,
          axis=1
      )
  )

  return times, rmse, nup_q, ndn_q, nup_e, ndn_e

In [ ]:
dt_values = [0.05, 0.10, 0.15, 0.20, 0.50]

summary = []

In [ ]:
for dt in dt_values:

  n_max = int(3.0/dt)

  times, rmse, nup_q, ndn_q, nup_e, ndn_e = run_simulation(dt, n_max)

  summary.append({

      "dt": dt,
      "RMSE_final": rmse[-1],
      "RMSE_mid": rmse[len(rmse)//2]

  })

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(summary)
"""
# Format specific columns to scientific notation (e.g., 2 decimal places in 'e' format)
df["RMSE_final"] = df_sci["RMSE_final"].apply(lambda x: f"{x:.2e}")
df["RMSE_mid"] = df_sci["RMSE_mid"].apply(lambda x: f"{x:.2e}")
"""
print(df)

In [ ]:
import matplotlib.pyplot as plt

# 1. Set up figure container
fig, ax = plt.subplots(figsize=(6, len(df) * 0.5 + 1), dpi=300)
ax.axis('off')  # Turn off axes lines

# 2. Build the table from DataFrame contents
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc='center',
    loc='center'
)

# 3. Format text size and spacing
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

# 4. Optional: Add styling (dark blue header, alternate zebra rows)
header_color = '#1f4e78'
zebra_color = '#f2f6fa'

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#d0d7de')
    if row == 0:
        cell.set_facecolor(header_color)
        cell.get_text().set_color('white')
        cell.get_text().set_weight('bold')
    else:
        cell.set_facecolor(zebra_color if row % 2 == 0 else '#ffffff')
        cell.get_text().set_color('#222222')

# 5. Save table to PNG
plt.savefig('summary_table.png', bbox_inches='tight', dpi=300)
#PDF
plt.savefig('summary_table.pdf', bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(
    df["dt"],
    df["RMSE_final"],
    'o-'
)

plt.xlabel("Time Step (dt)", fontsize=14)
plt.ylabel("RMSE_final", fontsize=14)
plt.title("Trotter Error vs Time Step", fontsize=14.5)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.grid()

# 1. Save as PNG (dpi=300 for crisp quality, bbox_inches='tight' prevents edge cropping)
plt.savefig('trotter_error_plot.png', bbox_inches='tight', dpi=300)
plt.savefig('trotter_error_plot.pdf', bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
dt_plot = 0.1
n_max_plot = 30
times = np.arange(n_max_plot + 1) * dt_plot

N_tot  = nup_q.sum(axis=1) + ndn_q.sum(axis=1)
Sz_tot = 0.5 * (nup_q.sum(axis=1) - ndn_q.sum(axis=1))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(times, N_tot / L, 'o-', color='seagreen'); ax[0].axhline(1.0, ls='--', c='k', lw=.8)
ax[0].set_ylabel("filling  $N/L$"); ax[0].set_ylim(0.9, 1.1)
ax[1].plot(times, Sz_tot, 'o-', color='darkorange');  ax[1].axhline(0.0, ls='--', c='k', lw=.8)
ax[1].set_ylabel(r"total $S^z$"); ax[1].set_ylim(-0.1, 0.1)
for a in ax: a.set_xlabel(r"$t\,/\,t_h^{-1}$ "); a.grid(alpha=.3)

fig.suptitle(
    "Particle Number and Spin Conservation During the 1D Fermi-Hubbard Quench",
    fontsize=14.5
)


fig.tight_layout();

# Save figure as PNG
plt.savefig(
    "particle_number_spin_conservation.png",
    bbox_inches="tight",
    dpi=300
)

# Save figure as PDF
plt.savefig(
    "particle_number_spin_conservation.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

print(f"filling drift over the whole run: {np.abs(N_tot/L - 1).max():.2e}")
print(f"S^z    drift over the whole run: {np.abs(Sz_tot).max():.2e}")